<div class="title-slide">

# Make the Most of What You’ve Got: Deep Learning Optimization with PyTorch Profiler and Holistic Trace Analysis 

## Collin Wilson

</div>

# Getting started with `sn-torch-lab`

### This talk is given in a Jupyter notebook format. You can download it and all the materials to recreate the demo on GitHub:

```bash
git clone https://github.com/cwilson51/sn-torch-lab.git
```

### You can then install the repository as a package in a virtual environment. On Alliance Clusters:

```bash
module load StdEnv/2023 python/3.11 scipy-stack/2026a
virtualenv --no-download torch-labENV && source torch-labENV/bin/activate
pip install --no-index upgrade pip

git clone https://github.com/cwilson51/sn-torch-lab.git
pip install --no-index ./sn-torch-lab
```

<div class="pt-slide">
    
# What we'll cover today

- ## PyTorch Profiler revisited
    - ### Inline profiling
    - ### Generating traces
- ## Examining traces and simple optimizations
- ## Holistic trace analysis (HTA)
    - ### Single GPU
    - ### Distributed workloads
- ## Trace Analysis with Brine
</div>

<div class="pt-slide">
    
# PyTorch Profiler

- ## Built-in module of PyTorch
- ## Can profile run times, memory and generate detailed traces of PyTorch code
    - ### see layer-by-layer breakdown of model performance
- ## Both CPU and GPU profiling
- ## Code modification required - wrap code with a context manager
- ## Works for NVIDIA and AMD GPUs; separate tool for Apple MPS
- ## Today’s live profiles are with the H100 1g.10gb MIG flavour on Nibi
</div>

# Primary components of PyTorch profiler code

```python
from torch.profiler import ProfilerActivity, profile, record_function, schedule
```
- ### `ProfilerActivity` - we must pass the profiler a list of activities to profile. Here we'll profile the CPU and GPU if available
- ### `profile` - the profile object used to create and store the profile during execution
- ### `record_function` - context manager we can use to label sections of code . e.g 'training' and 'inference' or 'forward-pass' and 'backward-pass'
- ### `schedule` - schedule is used to control generating profiles over several iterations, we'll see more on this later

# Working example

## ResNet50 on random inputs

In [10]:
import torch
from torch.utils.data import DataLoader
from torch.profiler import ProfilerActivity, profile, record_function, schedule

from torchvision.models import resnet50
from torchvision.datasets import FakeData
from torchvision import transforms

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

activities = [ProfilerActivity.CPU] 
if torch.cuda.is_available():
    device = "cuda"
    activities += [ProfilerActivity.CUDA]

### Model and optimizer

In [5]:
model = resnet50()
model.to(device)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

### Data

In [11]:
train_ds = FakeData(
    size=10_000,                 # number of samples
    image_size=(3, 224, 224),
    num_classes=1000,
    transform=transforms.ToTensor(),
)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,
                          num_workers=4, pin_memory=True)

### Run a simple profile

In [16]:
inputs, labels = next(iter(train_loader))
inputs, labels = inputs.to(device), labels.to(device)

with profile(activities=activities, record_shapes=True) as prof:

    model.train()
    with record_function("model_training"): # record_function can be used to get an aggregate measure of code sections and label them in traces
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        
    model.eval()
    with record_function("model_inference"):
        model(inputs)

    prof.step() # !!! Step the profiler !!!

USDT:2026-11-17 15:54:44 2270237:2270237 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-11-17 15:54:45 2270237:2270237 SyncActivityProfilerHandler.cpp:46] profiler_stop


### Results:

In [10]:
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                         model_training        32.48%        3.060s        66.12%        6.230s        6.230s       0.000us         0.00%      88.278ms      88.278ms             1  
                                           aten::conv2d         0.00%     215.281us        31.95%        3.010s      28.397ms       0.000us         0.00%      72.990ms     688.580us           106  
         

## Reduced for readability:
```
-------------------------------------------------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CUDA   Self CUDA %   CUDA time avg   # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  
                                         model_training       0.000us         0.00%      88.278ms             1  
                                           aten::conv2d       0.000us         0.00%     688.580us           106  
                                      aten::convolution       0.000us         0.00%     688.580us           106  
                                     aten::_convolution       0.000us         0.00%     688.580us           106  
                                aten::cudnn_convolution      72.990ms        22.33%     688.580us           106  
autograd::engine::evaluate_function: CudnnBatchNormB...       0.000us         0.00%     736.648us            53  
                                CudnnBatchNormBackward0       0.000us         0.00%     736.648us            53  
                        aten::cudnn_batch_norm_backward      39.042ms        11.94%     736.648us            53  
autograd::engine::evaluate_function: ConvolutionBack...       0.000us         0.00%       1.795ms            53  
                                   ConvolutionBackward0       0.000us         0.00%       1.590ms            53  
-------------------------------------------------------   ------------  ------------  ------------  ------------  
Self CPU time total: 9.422s
Self CUDA time total: 326.916ms
```


# Exporting traces

- ## Profiles are most useful as traces that can be analyzed with other tools
- ## We can export traces as .JSON files with `prof.export_chrome_trace("trace.json")`

In [17]:
prof.export_chrome_trace("example-traces/live/trace.json")

In [20]:
! ls -lh example-traces/live

total 3.8M
-rw-r----- 1 c7wilson def-c7wilson 3.8M Nov 17 15:54 trace.json


# `schedule`

- ## When profiling longer training runs, profiles can get very large.
- ## We can capture select iterations using a `schedule`:
```python
               my_schedule = schedule(skip_first=10,
                                      wait=5,
                                      warmup=5,
                                      active=5,
                                      repeat=1)

```
- ## Profiler skips the first `skip_first` steps, then waits `wait` steps
- ## Starting the profile can skew results, so the profiler starts and does a `warmup`
- ## The profiler is active for `active` steps
- ## The profiler can repeat a cycle of `wait` + `warmup` + `active` as set by the user with `repeat`

In [23]:
my_schedule = schedule(skip_first=0,
                       wait=1,
                       warmup=2,
                       active=2,
                       repeat=2)


with profile(activities=activities, schedule=my_schedule, record_shapes=True) as prof:

    model.train()
    for _ in range(10):
    
        inputs, labels = next(iter(train_loader))
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        prof.step()


USDT:2026-11-17 16:02:20 2270237:2270237 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-11-17 16:02:21 2270237:2270237 SyncActivityProfilerHandler.cpp:46] profiler_stop
USDT:2026-11-17 16:02:22 2270237:2270237 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-11-17 16:02:23 2270237:2270237 SyncActivityProfilerHandler.cpp:46] profiler_stop


# Using trace handlers

- ## As we've seen, the profile only holds one cycle at a time
- ## We can define a trace handler to print results and/or export traces after each cycle
- ## The trace handler function is passed to the profiler using the `on_trace_ready` argument

## Example:

In [34]:
sort_by_keyword = "self_" + device + "_time_total"

def trace_handler(p):
    output = p.key_averages().table(sort_by=sort_by_keyword, row_limit=1).splitlines()[-2:]
    print(output)
    p.export_chrome_trace(f"example-traces/live/trace_{p.step_num}.json")

with profile(activities=activities, schedule=my_schedule, record_shapes=True, on_trace_ready=trace_handler) as prof:

    model.train()
    for _ in range(10):
    
        inputs, labels = next(iter(train_loader))
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        prof.step()

USDT:2026-11-17 16:12:02 2270237:2270237 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-11-17 16:12:03 2270237:2270237 SyncActivityProfilerHandler.cpp:46] profiler_stop


['Self CPU time total: 873.733ms', 'Self CUDA time total: 726.612ms']


USDT:2026-11-17 16:12:05 2270237:2270237 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-11-17 16:12:05 2270237:2270237 SyncActivityProfilerHandler.cpp:46] profiler_stop


['Self CPU time total: 893.256ms', 'Self CUDA time total: 726.481ms']


In [29]:
! ls -lh example-traces/live

total 18M
-rw-r----- 1 c7wilson def-c7wilson 6.7M Nov 17 16:05 trace_10.json
-rw-r----- 1 c7wilson def-c7wilson 6.7M Nov 17 16:05 trace_5.json
-rw-r----- 1 c7wilson def-c7wilson 3.8M Nov 17 15:54 trace.json


# Examining traces and simple optimizations

## Traces

- ## JSON traces are viewable at [ui.perfetto.dev](ui.perfetto.dev)
- ## The traces we'll view are all single GPU traces with ResNet50 on a synthetic image dataset designed to mimic ImageNet using an H100 1g_10GB MIG on Nibi
- ## Traces can be downloaded from [nextcloud](https://nextcloud.sharcnet.ca/s/smYAE6DynAEQWpn)
- ## If you've installed this repo, you can also generate the single-GPU examples on your own system or on a cluster with
```bash
torch-lab generate-example-traces
```

# Examining traces and simple optimizations

## Optimizations

- ## The example traces cover several simple optimizations
    - ### Serial, `num_workers` unset
    - ### Serial, `num_workers=1`
    - ### Threaded `num_workers=2`
    - ### Threaded, compiled with `torch.compile`
    - ### Threaded, compiled and using Automatic Mixed Precision
- ## We don't have time to cover these optimizations in detail, but if you are interested, you can attend my PyTorch course as part of SHARCNET's Intro ARC series at the end of February. Sign up at training.sharcnet.ca! 
